## Data Loading

In [1]:
# ── Setup: imports, FBref session, cache dir, table parser ──────────────────
# Run this first. The cells below depend on every name defined here.
import os
import pandas as pd
import soccerdata as sd
from lxml import etree, html          # html.parse(...) and etree.HTMLParser/fromstring
from pathlib import Path

# soccerdata's internal MultiIndex-aware table parser (used on raw <table> elements)
from soccerdata.fbref import _parse_table

# Where scraped HTML pages get cached (same location build_default_xmins.py uses)
CACHE_DIR = Path.home() / "soccerdata" / "data" / "FBref" / "intl_probe"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Session-only FBref reader. leagues/seasons are required by the constructor but
# irrelevant here — we call fbref.get(url, cache_path) with explicit comp URLs.
fbref = sd.FBref(leagues=["ENG-Premier League"], seasons=2025)

print(f"Setup OK · CACHE_DIR={CACHE_DIR}")

[06/03/26 15:50:28] INFO     No custom team name replacements found. You can configure these in       ]8;id=6183193;file:///opt/miniconda3/envs/world-cup-fantasy/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=6183194;file:///opt/miniconda3/envs/world-cup-fantasy/lib/python3.11/site-packages/soccerdata/_config.py#92\92]8;;\
                             /Users/fungs4/soccerdata/config/teamname_replacements.json.                           

                    INFO     Custom league dict loaded from                                          ]8;id=6183200;file:///opt/miniconda3/envs/world-cup-fantasy/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=6183201;file:///opt/miniconda3/envs/world-cup-fantasy/lib/python3.11/site-packages/soccerdata/_config.py#188\188]8;;\
                             /Users/fungs4/soccerdata/config/league_dict.json.                                     

                    INFO     Saving cached data to /Users/fungs4/soccerdata/data/FBref               ]8;id=6183208;file:///opt/miniconda3/envs/world-cup-fantasy/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=6183209;file:///opt/miniconda3/envs/world-cup-fantasy/lib/python3.11/site-packages/soccerdata/_common.py#250\250]8;;\

dlopen /usr/local/bin/../Frameworks/Google Chrome for Testing Framework.framework/Versions/138.0.7204.168/Google Chrome for Testing Framework: dlopen(/usr/local/bin/../Frameworks/Google Chrome for Testing Framework.framework/Versions/138.0.7204.168/Google Chrome for Testing Framework, 0x0105): tried: '/usr/local/bin/../Frameworks/Google Chrome for Testing Framework.framework/Versions/138.0.7204.168/Google Chrome for Testing Framework' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/usr/local/bin/../Frameworks/Google Chrome for Testing Framework.framework/Versions/138.0.7204.168/Google Chrome for Testing Framework' (no such file), '/usr/local/bin/../Frameworks/Google Chrome for Testing Framework.framework/Versions/138.0.7204.168/Google Chrome for Testing Framework' (no such file).

Setup OK · CACHE_DIR=/Users/fungs4/soccerdata/data/FBref/intl_probe

In [2]:
# Proof-of-concept: scrape one confederation WCQ stats page directly.
import pandas as pd

URL = "https://fbref.com/en/comps/6/stats/WCQ----UEFA-M-Stats"
CACHE = CACHE_DIR / "wcq_uefa_2026.html"

reader = fbref.get(URL, CACHE)
tree = html.parse(reader)
parser = etree.HTMLParser(recover=True)

# The main player stats table on a comp's /stats/ page is usually `stats_standard`
# (no comp-id suffix because the page IS that comp). FBref often comments it out.
candidates = []
for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
    root = etree.fromstring(f"<root>{c.text}</root>", parser)
    candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))

# Dedupe by id
seen = set()
unique = []
for t in candidates:
    if t.get("id") in seen:
        continue
    seen.add(t.get("id"))
    unique.append(t)

print(f"Found {len(unique)} stats_standard table(s): {[t.get('id') for t in unique]}")

# Parse the first one
df = _parse_table(unique[0])
print(f"\nShape: {df.shape}")
print(f"\nTop-level column groups: {df.columns.get_level_values(0).unique().tolist()}")
print(f"\nLeaf column names: {df.columns.get_level_values(1).tolist()}")

# Inspect — what does the Squad/Nation column look like?
print(f"\nFirst 3 rows:")
print(df.head(3))

# Flatten so we can poke at it
df_flat = df.copy()
df_flat.columns = [b if (not a or a == b or a.startswith("Unnamed")) else f"{a}_{b}" for a, b in df_flat.columns]
df_flat = df_flat.reset_index(drop=True)

# Find the team-affiliation column
team_cols = [c for c in df_flat.columns if c.lower() in ("squad", "team", "nation")]
print(f"\nTeam-affiliation columns: {team_cols}")

if team_cols:
    tc = team_cols[0]
    print(f"\nUnique {tc} values ({df_flat[tc].nunique()}):")
    print(sorted(df_flat[tc].dropna().unique().tolist()))

print(f"\nTotal players: {len(df_flat)}")

Found 1 stats_standard table(s): ['stats_standard']


Shape: (1590, 24)


Top-level column groups: ['Unnamed: 0_level_0', 'Unnamed: 1_level_0', 'Unnamed: 2_level_0', 'Unnamed: 3_level_0', 'Unnamed: 4_level_0', 'Unnamed: 5_level_0', 'Playing Time', 'Performance', 'Per 90 Minutes', 'Unnamed: 23_level_0']


Leaf column names: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls', 'Ast', 'G+A', 'G-PK', 'G+A-PK', 'Matches']


First 3 rows:

  Unnamed: 0_level_0  Unnamed: 1_level_0 Unnamed: 2_level_0  \
                  Rk              Player                Pos   
0                  1  Bárður Á Reynatrøð                 GK   
1                  2      Thelo Aasgaard                 MF   
2                  3          Liel Abada                 DF   

  Unnamed: 3_level_0 Unnamed: 4_level_0 Unnamed: 5_level_0 Playing Time  \
               Squad                Age               Born           MP   
0      Faroe Islands                 26               2000            2   
1             Norway                 23               2002            4   
2             Israel                 24               2001            2   

                    ... Performance                 Per 90 Minutes        \
  Starts  Min  90s  ...          PK PKatt CrdY CrdR            Gls   Ast   
0      2  180  2.0  ...           0     0    0    0            0.0   0.0   
1      1  120  1.3  ...           1     1    0    0           3.75  0.75   
2   


Team-affiliation columns: ['Squad']


Unique Squad values (54):

['Albania', 'Andorra', 'Armenia', 'Austria', 'Azerbaijan', 'Belarus', 'Belgium', 'Bosnia-Herzegovina', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 'Denmark', 'England', 'Estonia', 'Faroe Islands', 'Finland', 'France', 'Georgia', 'Germany', 'Gibraltar', 'Greece', 'Hungary', 'Iceland', 'Israel', 'Italy', 'Kazakhstan', 'Kosovo', 'Latvia', 'Liechtenstein', 'Lithuania', 'Luxembourg', 'Malta', 'Moldova', 'Montenegro', 'N. Macedonia', 'Netherlands', 'Northern Ireland', 'Norway', 'Poland', 'Portugal', 'Rep. of Ireland', 'Romania', 'San Marino', 'Scotland', 'Serbia', 'Slovakia', 'Slovenia', 'Spain', 'Sweden', 'Switzerland', 'Türkiye', 'Ukraine', 'Wales']


Total players: 1590

In [3]:
# Find URLs for all other "FIFA World Cup Qualification" comps from FBref's master comps index.
leagues_html = Path.home() / "soccerdata" / "data" / "FBref" / "leagues.html"
tree = html.parse(str(leagues_html))

# Find all anchor tags with text matching "World Cup Qualification"
wcq_links = []
for a in tree.xpath("//a"):
    text = (a.text or "").strip()
    href = a.get("href", "")
    if "World Cup Qualification" in text and "Women" not in text:
        wcq_links.append((text, href))

print(f"Found {len(wcq_links)} WCQ links:")
for text, href in wcq_links:
    print(f"  {text:55s} → {href}")


Found 7 WCQ links:

  FIFA World Cup Qualification — Inter-confederation play-offs → /en/comps/255/history/FIFA-World-Cup-Qualification----Inter-confederation-play-offs-Seasons

  FIFA World Cup Qualification — CAF                      → /en/comps/2/history/WCQ----CAF-M-Seasons

  FIFA World Cup Qualification — CONCACAF                 → /en/comps/3/history/WCQ----CONCACAF-M-Seasons

  FIFA World Cup Qualification — CONMEBOL                 → /en/comps/4/history/WCQ----CONMEBOL-M-Seasons

  FIFA World Cup Qualification — OFC                      → /en/comps/5/history/WCQ----OFC-M-Seasons

  FIFA World Cup Qualification — UEFA                     → /en/comps/6/history/WCQ----UEFA-M-Seasons

  FIFA World Cup Qualification — AFC                      → /en/comps/7/history/WCQ----AFC-M-Seasons

In [4]:
# Scrape all 7 WCQ pages → one combined player-stats DataFrame for the 2026 cycle.
import re

CONFED_URLS = {
    "UEFA":      "https://fbref.com/en/comps/6/stats/WCQ----UEFA-M-Stats",
    "CAF":       "https://fbref.com/en/comps/2/stats/WCQ----CAF-M-Stats",
    "CONCACAF":  "https://fbref.com/en/comps/3/stats/WCQ----CONCACAF-M-Stats",
    "CONMEBOL":  "https://fbref.com/en/comps/4/stats/WCQ----CONMEBOL-M-Stats",
    "OFC":       "https://fbref.com/en/comps/5/stats/WCQ----OFC-M-Stats",
    "AFC":       "https://fbref.com/en/comps/7/stats/WCQ----AFC-M-Stats",
    "InterConf": "https://fbref.com/en/comps/255/stats/FIFA-World-Cup-Qualification----Inter-confederation-play-offs-Stats",
}

def fetch_and_parse_wcq(confed, url):
    cache = CACHE_DIR / f"wcq_{confed}_2026.html"
    reader = fbref.get(url, cache)
    tree = html.parse(reader)
    parser = etree.HTMLParser(recover=True)

    candidates = []
    for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
        root = etree.fromstring(f"<root>{c.text}</root>", parser)
        candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
    candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))

    seen, unique = set(), []
    for t in candidates:
        if t.get("id") in seen:
            continue
        seen.add(t.get("id"))
        unique.append(t)
    if not unique:
        return None
    df = _parse_table(unique[0])
    # Flatten MultiIndex columns
    df.columns = [b if (not a or a == b or str(a).startswith("Unnamed")) else f"{a}_{b}"
                  for a, b in df.columns]
    df = df.reset_index(drop=True)
    df["confederation"] = confed
    return df

frames = []
for confed, url in CONFED_URLS.items():
    df = fetch_and_parse_wcq(confed, url)
    if df is None:
        print(f"  {confed}: NO TABLE FOUND")
        continue
    print(f"  {confed}: {len(df)} player rows, {df['Squad'].nunique()} squads")
    frames.append(df)

wcq_all = pd.concat(frames, ignore_index=True)
print(f"\n=== Combined: {len(wcq_all)} player rows across {wcq_all['Squad'].nunique()} nations ===")
print(f"Columns: {wcq_all.columns.tolist()}")

# Cross-check coverage against our WC squads
wc_players = pd.read_csv("../data/processed/player_fixtures.csv")[["player", "team"]].drop_duplicates()
wc_nations = sorted(wc_players["team"].unique())
print(f"\nWC nations in our roster: {len(wc_nations)}")

wcq_nations = set(wcq_all["Squad"].dropna().unique())
present     = [n for n in wc_nations if n in wcq_nations]
missing     = [n for n in wc_nations if n not in wcq_nations]
print(f"Present in WCQ scrape: {len(present)} / {len(wc_nations)}")
print(f"Missing (likely name-mismatch — fix with a mapping): {missing}")

  UEFA: 1590 player rows, 54 squads

  CAF: 2100 player rows, 53 squads

  CONCACAF: 952 player rows, 32 squads

  CONMEBOL: 462 player rows, 10 squads

  OFC: 227 player rows, 11 squads

...


=== Combined: 6848 player rows across 206 nations ===

Columns: ['Rk', 'Player', 'Pos', 'Squad', 'Age', 'Born', 'Playing Time_MP', 'Playing Time_Starts', 'Playing Time_Min', 'Playing Time_90s', 'Performance_Gls', 'Performance_Ast', 'Performance_G+A', 'Performance_G-PK', 'Performance_PK', 'Performance_PKatt', 'Performance_CrdY', 'Performance_CrdR', 'Per 90 Minutes_Gls', 'Per 90 Minutes_Ast', 'Per 90 Minutes_G+A', 'Per 90 Minutes_G-PK', 'Per 90 Minutes_G+A-PK', 'Matches', 'confederation']


WC nations in our roster: 48

Present in WCQ scrape: 43 / 48

Missing (likely name-mismatch — fix with a mapping): ['Bosnia and Herzegovina', 'Cabo Verde', 'Canada', 'Mexico', 'USA']

In [5]:
# Verify Cape Verde / Cabo Verde mapping, then build the filtered international-stats table.

# Sanity: list all FBref Squad names that look like Cape Verde
print("Squad names containing 'verde' or 'cape':")
for s in sorted(wcq_all["Squad"].dropna().unique()):
    if "verde" in s.lower() or "cape" in s.lower():
        print(f"  {s}")

# Map our WC squad names → FBref Squad names. Only entries that actually differ.
WC_TO_FBREF_SQUAD = {
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Cabo Verde": "Cape Verde",
    # Add more here if other mismatches surface
}

# Apply mapping to our roster, filter wcq_all to our 48
wc_players["fbref_squad"] = wc_players["team"].map(WC_TO_FBREF_SQUAD).fillna(wc_players["team"])
wc_squads_fbref = set(wc_players["fbref_squad"].unique())

intl = wcq_all[wcq_all["Squad"].isin(wc_squads_fbref)].copy()

print(f"\nFiltered international-stats rows: {len(intl)}")
print(f"Squads represented: {intl['Squad'].nunique()} / 48")

# Per-team match/minute summary
summary = (
    intl.assign(min_=pd.to_numeric(intl["Playing Time_Min"].astype(str).str.replace(",", ""), errors="coerce"))
        .groupby("Squad")
        .agg(players=("Player", "nunique"),
             total_min=("min_", "sum"),
             max_mp=("Playing Time_MP", "max"))
        .sort_values("max_mp", ascending=False)
)
print(f"\nPer-team summary (top 10 by matches played):")
print(summary.head(10))
print(f"\nPer-team summary (bottom 10 by matches played):")
print(summary.tail(10))

# Which of the 48 still have zero rows (should be USA, Canada, Mexico)
covered = set(intl["Squad"].unique())
still_missing = [t for t in wc_squads_fbref if t not in covered]
print(f"\nStill missing (expected to be the 3 co-hosts): {still_missing}")

Squad names containing 'verde' or 'cape':

  Cape Verde


Filtered international-stats rows: 1653

Squads represented: 45 / 48


Per-team summary (top 10 by matches played):

                players  total_min  max_mp
Squad                                     
Iraq                 48      19650      20
Ecuador              44      17706      18
Brazil               60      17810      18
Colombia             43      17816      18
Argentina            35      17738      18
Uruguay              41      17820      17
Paraguay             46      17773      17
Qatar                56      17782      16
Korea Republic       52      15840      16
Jordan               37      15840      16


Per-team summary (bottom 10 by matches played):

             players  total_min  max_mp
Squad                                  
Austria           28       7919       8
Croatia           32       7920       8
England           32       7920       8
Switzerland       23       5940       6
France            30       5917       6
Spain             32       5940       6
Scotland          24       5940       6
Portugal          26       5908       6
Germany           30       5940       6
New Zealand       24       4950       5


Still missing (expected to be the 3 co-hosts): ['Canada', 'USA', 'Mexico']

In [6]:
out = "../data/processed/international_wcq_2026.csv"
intl.to_csv(out, index=False)
print(f"Wrote {len(intl)} rows → {out}")

Wrote 1653 rows → ../data/processed/international_wcq_2026.csv

## More Data!! 

We want to get more international football player level data, e.g. Copa America for South American nations, UEFA Nations League for UEFA nations....

In [7]:
# Discover URLs for the 5 v2 comps from FBref's master comps index.
search_terms = [
    "UEFA Nations League",
    "Gold Cup",
    "Africa Cup of Nations",
    "UEFA Euro",
    "Copa Am",
]

leagues_html = Path.home() / "soccerdata" / "data" / "FBref" / "leagues.html"
tree = html.parse(str(leagues_html))

print("Candidate links per search term:\n")
for term in search_terms:
    print(f"=== {term!r} ===")
    matches = []
    for a in tree.xpath("//a"):
        text = (a.text or "").strip()
        href = a.get("href", "")
        if term.lower() in text.lower() and "Women" not in text and "U-" not in text and "Youth" not in text:
            matches.append((text, href))
    if not matches:
        print("  (no matches)")
    for text, href in matches:
        print(f"  {text:55s} → {href}")
    print()

Candidate links per search term:


=== 'UEFA Nations League' ===

  UEFA Nations League                                     → /en/comps/677/UEFA-Nations-League-Stats

  UEFA Nations League                                     → /en/comps/677/history/UEFA-Nations-League-Seasons

  UEFA Nations League                                     → /en/comps/677/UEFA-Nations-League-Stats

...

=== 'Copa Am' ===

  CONMEBOL Copa América                                   → /en/comps/685/history/Copa-America-Seasons

  Copa América Femenina                                   → /en/comps/158/history/Copa-America-Femenina-Seasons

  CONMEBOL Copa América                                   → /en/comps/685/Copa-America-Stats

  Copa América Femenina                                   → /en/comps/158/Copa-America-Femenina-Stats

In [8]:
# v2: add 5 more international competitions, with a `competition` column for downstream filtering.

V2_COMPS = [
    ("UEFA",     "UEFA Nations League",   "https://fbref.com/en/comps/677/stats/UEFA-Nations-League-Stats"),
    ("CAF",      "AFCON 2025",            "https://fbref.com/en/comps/656/stats/Africa-Cup-of-Nations-Stats"),
    ("CONCACAF", "Gold Cup 2025",         "https://fbref.com/en/comps/681/stats/Gold-Cup-Stats"),
    ("UEFA",     "Euro 2024",             "https://fbref.com/en/comps/676/stats/UEFA-Euro-Stats"),
    ("CONMEBOL", "Copa America 2024",     "https://fbref.com/en/comps/685/stats/Copa-America-Stats"),
]

def fetch_comp_standard(url, cache_name):
    cache = CACHE_DIR / cache_name
    reader = fbref.get(url, cache)
    tree = html.parse(reader)
    parser = etree.HTMLParser(recover=True)

    candidates = []
    for c in tree.xpath("//comment()[contains(., 'stats_standard')]"):
        root = etree.fromstring(f"<root>{c.text}</root>", parser)
        candidates.extend(root.xpath(".//table[contains(@id, 'stats_standard')]"))
    candidates.extend(tree.xpath("//table[contains(@id, 'stats_standard')]"))
    seen, unique = set(), []
    for t in candidates:
        if t.get("id") in seen:
            continue
        seen.add(t.get("id"))
        unique.append(t)
    if not unique:
        return None, None

    tbl = unique[0]
    caption_el = tbl.xpath(".//caption")
    caption = "".join(caption_el[0].itertext()).strip() if caption_el else tbl.get("id", "")

    df = _parse_table(tbl)
    df.columns = [b if (not a or a == b or str(a).startswith("Unnamed")) else f"{a}_{b}"
                  for a, b in df.columns]
    df = df.reset_index(drop=True)
    return caption, df

v2_frames = []
for confed, comp, url in V2_COMPS:
    cache_name = f"{comp.replace(' ', '_')}.html"
    caption, df = fetch_comp_standard(url, cache_name)
    if df is None:
        print(f"  {comp}: NO TABLE FOUND ({url})")
        continue
    df["confederation"] = confed
    df["competition"] = comp
    print(f"  {comp:25s} | caption: {caption!r}")
    print(f"  {'':25s}   {len(df)} player rows, {df['Squad'].nunique()} squads")
    v2_frames.append(df)

v2_all = pd.concat(v2_frames, ignore_index=True)

# Add `competition` column to v1 (intl) so schemas align
intl["competition"] = intl["confederation"].map({
    "UEFA": "UEFA WCQ", "CAF": "CAF WCQ", "CONCACAF": "CONCACAF WCQ",
    "CONMEBOL": "CONMEBOL WCQ", "OFC": "OFC WCQ", "AFC": "AFC WCQ",
})

# Filter v2 to our 48 WC squads (reuse the mapping from v1)
v2_filtered = v2_all[v2_all["Squad"].isin(wc_squads_fbref)].copy()
print(f"\nv2 filtered to WC squads: {len(v2_filtered)} rows across {v2_filtered['Squad'].nunique()} squads")

# Stack v1 + v2
intl_all = pd.concat([intl, v2_filtered], ignore_index=True)
print(f"\n=== Combined v1+v2: {len(intl_all)} rows, {intl_all['Squad'].nunique()} squads ===")
print(intl_all.groupby("competition").size().sort_values(ascending=False))

# Co-host check — did we pick up USA/Canada/Mexico?
cohost_data = intl_all[intl_all["Squad"].isin(["United States", "USA", "Canada", "Mexico"])]
print(f"\nCo-host coverage now:")
print(cohost_data.groupby(["Squad", "competition"]).size())

  UEFA Nations League       | caption: 'Player Standard Stats 2024-2025 UEFA Nations League Table'

                              1457 player rows, 54 squads

  AFCON 2025                | caption: 'Player Standard Stats 2025 Africa Cup of Nations Table'

                              543 player rows, 24 squads

  Gold Cup 2025             | caption: 'Player Standard Stats 2025 Gold Cup Table'

...


v2 filtered to WC squads: 1248 rows across 36 squads


=== Combined v1+v2: 2901 rows, 47 squads ===

competition
UEFA Nations League    480
UEFA WCQ               470
AFC WCQ                409
CAF WCQ                382
Euro 2024              270
CONMEBOL WCQ           269
Copa America 2024      190
AFCON 2025             187
Gold Cup 2025          121
CONCACAF WCQ            99
OFC WCQ                 24
dtype: int64


Co-host coverage now:

Squad   competition      
Canada  Copa America 2024    22
        Gold Cup 2025        23
Mexico  Copa America 2024    20
        Gold Cup 2025        23
dtype: int64

In [9]:
# Check FBref's spelling for USA
print("Squad values containing 'unit' or 'state':")
for s in sorted(v2_all["Squad"].dropna().unique()):
    if "unit" in s.lower() or "state" in s.lower():
        print(f"  {s}")

# Add USA to the mapping and re-filter
WC_TO_FBREF_SQUAD["USA"] = "United States"

wc_players["fbref_squad"] = wc_players["team"].map(WC_TO_FBREF_SQUAD).fillna(wc_players["team"])
wc_squads_fbref = set(wc_players["fbref_squad"].unique())

# Re-filter both v1 and v2 with the updated mapping
intl_v1 = wcq_all[wcq_all["Squad"].isin(wc_squads_fbref)].copy()
intl_v1["competition"] = intl_v1["confederation"].map({
    "UEFA": "UEFA WCQ", "CAF": "CAF WCQ", "CONCACAF": "CONCACAF WCQ",
    "CONMEBOL": "CONMEBOL WCQ", "OFC": "OFC WCQ", "AFC": "AFC WCQ",
})
intl_v2 = v2_all[v2_all["Squad"].isin(wc_squads_fbref)].copy()

intl_all = pd.concat([intl_v1, intl_v2], ignore_index=True)
print(f"\nCombined v1+v2: {len(intl_all)} rows, {intl_all['Squad'].nunique()} squads")

# Co-host coverage now
cohost_data = intl_all[intl_all["Squad"].isin(["United States", "Canada", "Mexico"])]
print("\nCo-host coverage:")
print(cohost_data.groupby(["Squad", "competition"]).size())

# Save final v1+v2 international stats
out = "../data/processed/international_stats_2026.csv"
intl_all.to_csv(out, index=False)
print(f"\nWrote {len(intl_all)} rows → {out}")


Squad values containing 'unit' or 'state':

  United States


Combined v1+v2: 2945 rows, 48 squads


Co-host coverage:

Squad          competition      
Canada         Copa America 2024    22
               Gold Cup 2025        23
Mexico         Copa America 2024    20
               Gold Cup 2025        23
United States  Copa America 2024    21
               Gold Cup 2025        23
dtype: int64


Wrote 2945 rows → ../data/processed/international_stats_2026.csv

## Calculating Average Opponent Elo Strength

In [10]:
elo = pd.read_csv("../data/elo_ratings.csv")  # adjust path if different
print(elo.shape)
print(elo.columns.tolist())
print(elo.head(10))
print(f"\nUnique teams: {elo['team'].nunique() if 'team' in elo.columns else 'check column name'}")

(211, 92)

['Rank', 'Code', 'Country', 'PELE', '△ 1 year', '_2005Q0', '_2005Q1', '_2005Q2', '_2005Q3', '_2005Q4', '_2006Q1', '_2006Q2', '_2006Q3', '_2006Q4', '_2007Q1', '_2007Q2', '_2007Q3', '_2007Q4', '_2008Q1', '_2008Q2', '_2008Q3', '_2008Q4', '_2009Q1', '_2009Q2', '_2009Q3', '_2009Q4', '_2010Q1', '_2010Q2', '_2010Q3', '_2010Q4', '_2011Q1', '_2011Q2', '_2011Q3', '_2011Q4', '_2012Q1', '_2012Q2', '_2012Q3', '_2012Q4', '_2013Q1', '_2013Q2', '_2013Q3', '_2013Q4', '_2014Q1', '_2014Q2', '_2014Q3', '_2014Q4', '_2015Q1', '_2015Q2', '_2015Q3', '_2015Q4', '_2016Q1', '_2016Q2', '_2016Q3', '_2016Q4', '_2017Q1', '_2017Q2', '_2017Q3', '_2017Q4', '_2018Q1', '_2018Q2', '_2018Q3', '_2018Q4', '_2019Q1', '_2019Q2', '_2019Q3', '_2019Q4', '_2020Q1', '_2020Q2', '_2020Q3', '_2020Q4', '_2021Q1', '_2021Q2', '_2021Q3', '_2021Q4', '_2022Q1', '_2022Q2', '_2022Q3', '_2022Q4', '_2023Q1', '_2023Q2', '_2023Q3', '_2023Q4', '_2024Q1', '_2024Q2', '_2024Q3', '_2024Q4', '_2025Q1', '_2025Q2', '_2025Q3', '_2025Q4', '_2026Q1', '_2026

   Rank Code             Country    PELE  △ 1 year  _2005Q0  _2005Q1  _2005Q2  \
0     1  ESP        :es: Spain 🏆  2084.0       9.2      0.0    -10.0    -42.9   
1     2  ARG    :ar: Argentina 🏆  2067.2     -20.3      0.0     -8.7    -33.4   
2     3  ENG  :gb-eng: England 🏆  2029.8       9.0      0.0      0.3      4.1   
3     4  FRA       :fr: France 🏆  2025.8      19.4      0.0    -34.2    -35.3   
4     5  BRA       :br: Brazil 🏆  2001.2       2.5      0.0    -13.7    -28.9   
5     6  POR     :pt: Portugal 🏆  1968.8      -3.6      0.0    -15.1    -19.0   
6     7  GER      :de: Germany 🏆  1965.2     -18.2      0.0      2.2      2.5   
7     8  NED  :nl: Netherlands 🏆  1952.3       3.0      0.0     10.8     31.2   
8     9  COL     :co: Colombia 🏆  1945.1      10.0      0.0     -9.6     25.0   
9    10  NOR       :no: Norway 🏆  1932.8      64.2      0.0    -15.2      3.3   

   _2005Q3  _2005Q4  ...  _2024Q1  _2024Q2  _2024Q3  _2024Q4  _2025Q1  \
0    -58.3    -46.1  ...    -87.9  


Unique teams: check column name

In [11]:
import re

# === Build clean ELO lookup ===
elo = pd.read_csv("../data/elo_ratings.csv")
elo["country_clean"] = elo["Country"].apply(
    lambda s: re.sub(r":[a-z\-]+:\s*", "", str(s)).replace("🏆", "").strip()
)
print("Sample ELO country names:", elo["country_clean"].head(20).tolist())

# === Map FBref Squad names → ELO country names ===
# We need this for both wcq_all + v2_all (which together represent all comp participants).
FBREF_TO_ELO = {
    # Diacritic/punctuation differences
    "Bosnia-Herzegovina":   "Bosnia/Herzegovina",
    "Côte d'Ivoire":        "Cote d'Ivoire",
    "Curaçao":              "Curacao",
    # Abbreviations
    "Congo DR":             "Dem. Rep. Congo",
    "Congo":                "Rep, Congo",
    "Rep. of Ireland":      "Rep. Ireland",
    "N. Macedonia":         "North Macedonia",
    "UAE":                  "Unit. Arab Emir.",
    "China PR":             "China",
    "CAR":                  "Cent. Afr. Rep.",
    # Already-working entries (keep explicit for clarity)
    "Korea Republic":       "South Korea",
    "United States":        "United States",
    "IR Iran":              "Iran",
}

intl_full = pd.concat([wcq_all, v2_all], ignore_index=True)
intl_full["competition"] = intl_full["competition"].fillna(
    intl_full["confederation"].map({
        "UEFA": "UEFA WCQ", "CAF": "CAF WCQ", "CONCACAF": "CONCACAF WCQ",
        "CONMEBOL": "CONMEBOL WCQ", "OFC": "OFC WCQ", "AFC": "AFC WCQ",
    })
)
intl_full["elo_country"] = intl_full["Squad"].map(FBREF_TO_ELO).fillna(intl_full["Squad"])

# === Compute mean PELE per competition (over participating squads) ===
elo_lookup = elo.set_index("country_clean")["PELE"].to_dict()

participants = intl_full.groupby("competition")["elo_country"].unique().to_dict()
rows = []
unmatched = set()
for comp, squads in participants.items():
    pele_vals = []
    for s in squads:
        if s in elo_lookup:
            pele_vals.append(elo_lookup[s])
        else:
            unmatched.add(s)
    rows.append({"competition": comp, "n_squads": len(squads), "n_matched": len(pele_vals),
                 "mean_pele": sum(pele_vals)/len(pele_vals) if pele_vals else None})

comp_strength = pd.DataFrame(rows).sort_values("mean_pele", ascending=False)
print(f"\nUnmatched squads (need adding to FBREF_TO_ELO): {sorted(unmatched)}")
print(f"\nComp strength (raw mean PELE):")
print(comp_strength.to_string(index=False))

# Normalize: divide by max so the strongest comp = 1.0
max_pele = comp_strength["mean_pele"].max()
comp_strength["strength_mult"] = comp_strength["mean_pele"] / max_pele
print(f"\nComp strength (normalized):")
print(comp_strength[["competition", "mean_pele", "strength_mult"]].to_string(index=False))


Sample ELO country names:

['Spain', 'Argentina', 'England', 'France', 'Brazil', 'Portugal', 'Germany', 'Netherlands', 'Colombia', 'Norway', 'Uruguay', 'Ecuador', 'Senegal', 'Italy', 'Turkey', 'Belgium', 'Switzerland', 'Croatia', 'Japan', 'Denmark']


Unmatched squads (need adding to FBREF_TO_ELO): ['Antigua', 'British Virgin Islands', 'Chinese Taipei', 'Equ. Guinea', 'Guadeloupe', 'Korea DPR', 'Kyrgyz Republic', 'Papua NG', 'St. Lucia', 'St. Vincent', 'São Tomé', 'Trin & Tobago', 'Turks & Caicos', 'Türkiye', 'US Virgin Islands']


Comp strength (raw mean PELE):

        competition  n_squads  n_matched   mean_pele
       CONMEBOL WCQ        10         10 1864.430000
          Euro 2024        24         23 1848.347826
  Copa America 2024        16         16 1820.731250
UEFA Nations League        54         53 1682.900000
           UEFA WCQ        54         53 1682.900000
      Gold Cup 2025        16         14 1655.064286
         AFCON 2025        24         23 1652.873913
            CAF WCQ        53         51 1540.664706
       CONCACAF WCQ        32         25 1392.172000
            AFC WCQ        46         43 1346.158140
            OFC WCQ        11         10 1085.030000


Comp strength (normalized):

        competition   mean_pele  strength_mult
       CONMEBOL WCQ 1864.430000       1.000000
          Euro 2024 1848.347826       0.991374
  Copa America 2024 1820.731250       0.976562
UEFA Nations League 1682.900000       0.902635
           UEFA WCQ 1682.900000       0.902635
      Gold Cup 2025 1655.064286       0.887705
         AFCON 2025 1652.873913       0.886530
            CAF WCQ 1540.664706       0.826346
       CONCACAF WCQ 1392.172000       0.746701
            AFC WCQ 1346.158140       0.722021
            OFC WCQ 1085.030000       0.581963

In [12]:
# Save comp_strength so notebook 03 can pick it up
out = "../data/processed/comp_strength.csv"
comp_strength[["competition", "mean_pele", "strength_mult"]].to_csv(out, index=False)
print(f"Wrote {len(comp_strength)} comps → {out}")

Wrote 11 comps → ../data/processed/comp_strength.csv

## xMins Projections using International Data

In [13]:
# Competition reliability weights — downweight older/lower-stakes comps when
# aggregating minutes. Unlisted competitions fall back to 0.5 via .fillna(0.5).
COMP_WEIGHTS = {
    "UEFA WCQ": 1.0, "CAF WCQ": 1.0, "CONMEBOL WCQ": 1.0,
    "AFC WCQ": 1.0, "CONCACAF WCQ": 1.0, "OFC WCQ": 1.0,
    "UEFA Nations League": 0.7,
    "AFCON 2025": 0.7,
    "Gold Cup 2025": 0.7,
    "Euro 2024": 0.5,
    "Copa America 2024": 0.5,
}

In [14]:
# === Default xMins v4: hybrid normalization + mp_share ranking + WC-roster filter ===
import numpy as np
import unicodedata
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from data.manual_overrides import MANUAL_OVERRIDES
from data.manual_starting_xi import MANUAL_STARTING_XI

def to_ascii(name):
    s = str(name)
    # Turkish dotless ı/İ → regular i/I (must happen before NFKD)
    s = s.replace("ı", "i").replace("İ", "I")
    for ch in ("'", "'", "`", "ʼ"):
        s = s.replace(ch, "")
    return unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode().lower().strip()


# ─── Load + per-player aggregates ───────────────────────────────────────────
intl = pd.read_csv("../data/processed/international_stats_2026.csv")
intl["min_num"] = pd.to_numeric(intl["Playing Time_Min"].astype(str).str.replace(",", ""), errors="coerce").fillna(0)
intl["mp_num"]  = pd.to_numeric(intl["Playing Time_MP"], errors="coerce").fillna(0)
intl["comp_w"]  = intl["competition"].map(COMP_WEIGHTS).fillna(0.5)

team_matches = (intl.groupby(["Squad", "competition"])["mp_num"].max().reset_index()
                .rename(columns={"mp_num": "team_matches"}))
team_matches["comp_w"] = team_matches["competition"].map(COMP_WEIGHTS).fillna(0.5)
team_matches["weighted_team_mp"] = team_matches["team_matches"] * team_matches["comp_w"]
team_avail = (team_matches.groupby("Squad")["weighted_team_mp"].sum()
              .reset_index().rename(columns={"weighted_team_mp": "team_mp_w"}))

intl["weighted_min"] = intl["min_num"] * intl["comp_w"]
intl["weighted_mp"]  = intl["mp_num"]  * intl["comp_w"]

player_agg = (intl.groupby(["Player", "Squad"], as_index=False)
              .agg(weighted_min=("weighted_min", "sum"),
                   weighted_mp=("weighted_mp", "sum")))

player_pos = (intl.groupby(["Player", "Squad"])["Pos"]
              .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else x.iloc[0])
              .reset_index())
player_agg = player_agg.merge(player_pos, on=["Player", "Squad"]).merge(team_avail, on="Squad")
player_agg["is_gk"]            = player_agg["Pos"].str.startswith("GK")
player_agg["mp_share"]         = player_agg["weighted_mp"] / player_agg["team_mp_w"]

K_COND_SHRINK = 5      # phantom appearances toward prior
PRIOR_COND    = 75     # typical starter minutes-per-appearance

player_agg["conditional_min_raw"] = np.where(
    player_agg["weighted_mp"] > 0,
    player_agg["weighted_min"] / player_agg["weighted_mp"], 0
)
player_agg["conditional_min"] = np.where(
    player_agg["weighted_mp"] > 0,
    (player_agg["weighted_min"] + K_COND_SHRINK * PRIOR_COND) /
    (player_agg["weighted_mp"] + K_COND_SHRINK),
    0
)
player_agg["per_team_match"]   = player_agg["weighted_min"] / player_agg["team_mp_w"]
player_agg["name_ascii"]       = player_agg["Player"].apply(to_ascii)

# ─── WC roster filter ───────────────────────────────────────────────────────
WC_TO_FBREF_SQUAD = {
    "Bosnia and Herzegovina": "Bosnia-Herzegovina",
    "Cabo Verde": "Cape Verde",
    "USA": "United States",
}
wc_roster = pd.read_csv("../data/processed/player_fixtures.csv")[["player", "team", "position"]].drop_duplicates()
wc_roster["name_ascii"] = wc_roster["player"].apply(
    lambda p: to_ascii(MANUAL_OVERRIDES.get(p, p))
)
wc_roster["fbref_squad"] = wc_roster["team"].map(WC_TO_FBREF_SQUAD).fillna(wc_roster["team"])

roster_keys = set(zip(wc_roster["name_ascii"], wc_roster["fbref_squad"]))
player_agg["roster_key"] = list(zip(player_agg["name_ascii"], player_agg["Squad"]))
player_agg["in_roster"]  = player_agg["roster_key"].isin(roster_keys)

print(f"Intl player rows: {len(player_agg)}")
print(f"  - in WC roster: {player_agg['in_roster'].sum()}")
print(f"  - NOT in roster (excluded): {(~player_agg['in_roster']).sum()}")

# ─── Pass 2 (enhanced): multi-scorer fuzzy match within-team ────────────────
from rapidfuzz import process, fuzz

matched_keys_so_far = set(zip(player_agg[player_agg["in_roster"]]["name_ascii"],
                              player_agg[player_agg["in_roster"]]["Squad"]))
roster_unmatched = wc_roster[~wc_roster.apply(
    lambda r: (r["name_ascii"], r["fbref_squad"]) in matched_keys_so_far, axis=1
)].copy()
unclaimed_intl = player_agg[~player_agg["in_roster"]][["Squad", "name_ascii"]].copy()

# Multiple scorers, each with its own threshold. Player matches if ANY passes.
# Within-team matching keeps false-positive risk low.
SCORERS = [
    ("token_sort_ratio", 80),  # was 80; relaxed for word-order/spelling variants
    ("WRatio",            85), # rapidfuzz's weighted heuristic — handles abbreviation
    ("partial_ratio",     90), # high threshold to require strong substring overlap
]

fuzzy_hits = []
match_log = []  # for inspection: (roster_name, matched_intl_name, scorer, score)

for _, r in roster_unmatched.iterrows():
    pool = unclaimed_intl[unclaimed_intl["Squad"] == r["fbref_squad"]]["name_ascii"].tolist()
    if not pool:
        continue

    best_match = None
    best_score = -1
    best_scorer = None
    for scorer_name, threshold in SCORERS:
        scorer = getattr(fuzz, scorer_name)
        result = process.extractOne(r["name_ascii"], pool,
                                     scorer=scorer, score_cutoff=threshold)
        if result:
            intl_ascii, score, _ = result
            if score > best_score:
                best_score = score
                best_match = intl_ascii
                best_scorer = scorer_name

    if best_match:
        fuzzy_hits.append((best_match, r["fbref_squad"]))
        match_log.append((r["player"], best_match, best_scorer, best_score))
        unclaimed_intl = unclaimed_intl[
            ~((unclaimed_intl["Squad"] == r["fbref_squad"]) &
              (unclaimed_intl["name_ascii"] == best_match))
        ]

print(f"Multi-scorer fuzzy matches: {len(fuzzy_hits)} (was 78 with single scorer)")
roster_keys.update(fuzzy_hits)
player_agg["in_roster"] = player_agg["roster_key"].isin(roster_keys)
print(f"Total matched after enhanced fuzzy: {player_agg['in_roster'].sum()}")

# Quick audit: top fuzzy hits by scorer (so you can spot suspicious matches)
import pandas as pd
log_df = pd.DataFrame(match_log, columns=["roster_name", "matched_intl", "scorer", "score"])
print(f"\nMatches by scorer:\n{log_df['scorer'].value_counts()}")
print(f"\nLowest-confidence matches (potential false positives, score < 80):")
print(log_df[log_df['score'] < 80].sort_values('score').head(15).to_string(index=False))


squad_agg = player_agg[player_agg["in_roster"]].copy()

# ─── Starter selection: min mp_share eligibility + per_team_match ranking ──
MIN_STARTER_MP_SHARE = 0.4

squad_agg["starter_eligible"] = squad_agg["mp_share"] >= MIN_STARTER_MP_SHARE
squad_agg["rank_in_pool"] = (
    squad_agg[squad_agg["starter_eligible"]]
    .groupby(["Squad", "is_gk"])["per_team_match"]
    .rank(method="first", ascending=False)
)
squad_agg["rank_in_pool"] = squad_agg["rank_in_pool"].fillna(999)
squad_agg["is_starter"] = (
    squad_agg["starter_eligible"] & (
        ((squad_agg["is_gk"]) & (squad_agg["rank_in_pool"] == 1)) |
        ((~squad_agg["is_gk"]) & (squad_agg["rank_in_pool"] <= 10))
    )
)


# ─── Manual XI overrides where defined (else keep algorithmic is_starter) ──
MANUAL_XI_ASCII = {
    team: {to_ascii(MANUAL_OVERRIDES.get(p, p)) for p in players}
    for team, players in MANUAL_STARTING_XI.items()
}
FBREF_TO_WC = {v: k for k, v in WC_TO_FBREF_SQUAD.items()}
squad_agg["wc_team"] = squad_agg["Squad"].map(FBREF_TO_WC).fillna(squad_agg["Squad"])

def _apply_manual_xi(row):
    team = row["wc_team"]
    if team in MANUAL_XI_ASCII:
        return row["name_ascii"] in MANUAL_XI_ASCII[team]
    return row["is_starter"]   # algorithmic fallback for teams without manual XI

squad_agg["is_starter"] = squad_agg.apply(_apply_manual_xi, axis=1)

# Audit: count how many manual-XI players matched per team
print("Manual XI match counts (should be 11 each for the 24 teams):")
for team, ascii_set in MANUAL_XI_ASCII.items():
    fbref_sq = WC_TO_FBREF_SQUAD.get(team, team)
    matched = squad_agg[(squad_agg["wc_team"] == team) & squad_agg["is_starter"]].shape[0]
    print(f"  {team:25s} {matched}/11")



# ─── Hybrid normalization (starters → conditional; non-starters → leftover) ─
starter_sum_gk = (squad_agg[squad_agg["is_gk"] & squad_agg["is_starter"]]
                  .groupby("Squad")["conditional_min"].sum().rename("starter_sum_gk").reset_index())
starter_sum_of = (squad_agg[~squad_agg["is_gk"] & squad_agg["is_starter"]]
                  .groupby("Squad")["conditional_min"].sum().rename("starter_sum_of").reset_index())
squad_agg = (squad_agg.merge(starter_sum_gk, on="Squad", how="left")
                       .merge(starter_sum_of, on="Squad", how="left"))
squad_agg["starter_sum"] = np.where(squad_agg["is_gk"], squad_agg["starter_sum_gk"], squad_agg["starter_sum_of"])
squad_agg["pool_budget"] = np.where(squad_agg["is_gk"], 90, 900)
squad_agg["starter_factor"] = np.minimum(1.0, squad_agg["pool_budget"] / squad_agg["starter_sum"])

ns_sum_gk = (squad_agg[squad_agg["is_gk"] & ~squad_agg["is_starter"]]
             .groupby("Squad")["per_team_match"].sum().rename("ns_sum_gk").reset_index())
ns_sum_of = (squad_agg[~squad_agg["is_gk"] & ~squad_agg["is_starter"]]
             .groupby("Squad")["per_team_match"].sum().rename("ns_sum_of").reset_index())
squad_agg = (squad_agg.merge(ns_sum_gk, on="Squad", how="left")
                       .merge(ns_sum_of, on="Squad", how="left"))
squad_agg["ns_sum"] = np.where(squad_agg["is_gk"], squad_agg["ns_sum_gk"], squad_agg["ns_sum_of"])
squad_agg["ns_sum"] = squad_agg["ns_sum"].fillna(0)
squad_agg["ns_budget"] = squad_agg["pool_budget"] - squad_agg["starter_sum"] * squad_agg["starter_factor"]
squad_agg["ns_factor"] = np.where(
    squad_agg["ns_sum"] > 0,
    np.minimum(1.0, squad_agg["ns_budget"] / squad_agg["ns_sum"]),
    0
)

squad_agg["xmins"] = np.where(
    squad_agg["is_starter"],
    squad_agg["conditional_min"] * squad_agg["starter_factor"],
    squad_agg["per_team_match"] * squad_agg["ns_factor"]
)

# ─── Build full output (incl unmatched roster) + even leftover distribution ─
# Map FBref Pos ("DF", "MF", "FW", "GK", "DF,MF" etc.) to our roster's position ("DEF", "MID", "FWD", "GK")
POS_MAP = {"GK": "GK", "DF": "DEF", "MF": "MID", "FW": "FWD"}

# Explode FBref's composite positions ("FW,MF" → two rows: FW + MF), map to roster format
squad_agg_exp = squad_agg.assign(
    pos_tokens=squad_agg["Pos"].str.split(",")
).explode("pos_tokens")
squad_agg_exp["pos_primary"] = squad_agg_exp["pos_tokens"].map(POS_MAP)
squad_agg_exp = squad_agg_exp.drop_duplicates(subset=["Player", "Squad", "pos_primary"])

# Merge on (name, squad, position) — composite-position players match against EITHER token
output = wc_roster[["player", "team", "position", "name_ascii", "fbref_squad"]].merge(
    squad_agg_exp[["name_ascii", "Squad", "pos_primary", "Pos", "is_starter", "xmins"]],
    left_on=["name_ascii", "fbref_squad", "position"],
    right_on=["name_ascii", "Squad", "pos_primary"],
    how="left"
).drop_duplicates(subset=["player", "team", "position"], keep="first")

output["xmins"]      = output["xmins"].fillna(0)
output["is_starter"] = output["is_starter"].fillna(False)

matched = (output["xmins"] > 0).sum()
print(f"Position-aware match (composite-expanded): {matched} / {len(output)} matched")



# Distribute leftover budget evenly across squad (capped at 90 per player)
team_total = output.groupby("team")["xmins"].sum().rename("team_total").reset_index()
team_size  = output.groupby("team").size().rename("team_size").reset_index()
output = output.merge(team_total, on="team").merge(team_size, on="team")
output["leftover"]    = (990 - output["team_total"]).clip(lower=0)
output["even_boost"]  = output["leftover"] / output["team_size"]
output["xmins"]       = (output["xmins"] + output["even_boost"]).clip(upper=90)

# Sanity check: team totals after redistribution
print("Per-team total xMins after even-distribution fallback:")
totals_after = output.groupby("team")["xmins"].sum().sort_values()
print(totals_after.head()); print("..."); print(totals_after.tail())

# Spot-check the affected teams
for sq in ["Brazil", "Egypt", "Jordan", "England", "Spain"]:
    print(f"\n{sq} top 15 by xMins after redistribution:")
    df = output[output["team"] == sq].nlargest(15, "xmins")
    cols = ["player", "Pos", "is_starter", "xmins"]
    print(df[cols].round(2).to_string(index=False))

Intl player rows: 1944

  - in WC roster: 1040

  - NOT in roster (excluded): 904

Multi-scorer fuzzy matches: 19 (was 78 with single scorer)

Total matched after enhanced fuzzy: 1059

...

               player   Pos is_starter  xmins
       Abdallah Nasib    DF       True  89.34
        Yazan Al Arab    DF       True  89.29
     Yazeed  Abulaila    GK       True  88.10
         Ehsan Haddad MF,DF       True  87.51
            Ali Olwan FW,MF       True  87.51
     Nizar Al Rashdan    MF       True  82.40
      Mousa Al Tamari FW,MF       True  81.73
     Noor Al Rawabdeh    MF       True  79.70
 Mohammad Abu Al Nadi    DF       True  76.96
     Mahmoud Al Mardi MF,FW       True  74.71
Mohammad Abu Hasheesh MF,DF      False  31.97
        Ibrahim Sadeh    MF      False  25.51
         Raja'ei Ayed    MF      False  20.69
   Husam Abu Al Dahab    DF      False  18.18
         Saleem Obaid    DF      False  11.14


England top 15 by xMins after redistribution:

         player   Pos is_starter  xmins
Jordan Pickford    GK       True  88.26
     Marc Guéhi    DF       True  81.65
    John Stones    DF       True  81.44
Jude Bellingham MF,FW       True  80.90
     Harry Kane    FW       True  80.43
  Nico O'Reilly    DF       True  79.92
    Declan Rice    MF       True  76.79
Elliot Anderson    MF       True  75.13
    Bukayo Saka MF,FW       True  75.08
    Reece James    DF       True  71.63
Marcus Rashford    MF       True  60.22
     Ezri Konsa    DF      False  27.47
  Morgan Rogers FW,MF      False  18.79
 Anthony Gordon MF,FW      False  17.58
       Dan Burn    DF      False  15.74


Spain top 15 by xMins after redistribution:

          player   Pos is_starter  xmins
  Marc Cucurella    DF       True  90.00
      Unai Simón    GK       True  90.00
 Aymeric Laporte    DF       True  88.74
     Fabián Ruiz    MF       True  80.10
     Pau Cubarsí    DF       True  75.36
           Rodri    MF       True  71.96
 Marcos Llorente    DF       True  70.18
 Mikel Oyarzabal FW,MF       True  69.37
           Pedri    MF       True  64.70
   Ferran Torres    FW       True  62.35
Martín Zubimendi    MF      False  43.01
    Mikel Merino    MF      False  36.10
     Pedro Porro    DF      False  32.58
       Dani Olmo    MF      False  24.49
      Álex Baena FW,MF      False  18.69

Export xMins

In [15]:
# === Export default_xmins.csv for build_projections.py ===
default_xmins = output[["player", "team", "position", "xmins"]].rename(
    columns={"xmins": "default_xmins"}
)
default_xmins["default_xmins"] = default_xmins["default_xmins"].round(2)

out_path = "../data/default_xmins.csv"
default_xmins.to_csv(out_path, index=False)

print(f"Wrote {len(default_xmins)} rows → {out_path}")
print(f"\nSample:")
print(default_xmins.head(10).to_string(index=False))

# Per-team summary (sanity check)
print(f"\nTeam total xMins summary (5 lowest, 5 highest):")
team_summary = (default_xmins.groupby("team")["default_xmins"]
                .agg(team_total="sum", n_players="count")
                .sort_values("team_total"))
print(team_summary.head())
print("...")
print(team_summary.tail())

# Distribution check
print(f"\nxMins distribution:")
print(default_xmins["default_xmins"].describe().round(2))

Wrote 1256 rows → ../data/default_xmins.csv


Sample:

          player    team position  default_xmins
 Rayan Aït-Nouri Algeria      DEF          86.69
 Ramy Bensebaini Algeria      DEF          90.00
     Aïssa Mandi Algeria      DEF          87.91
Zinéddine Belaïd Algeria      DEF          15.57
  Rafik Belghali Algeria      DEF          82.80
    Achref Abada Algeria      DEF           7.23
 Mohammed Amoura Algeria      FWD          84.45
    Amine Gouiri Algeria      FWD          71.51
    Riyad Mahrez Algeria      MID          75.66
    Ibrahim Maza Algeria      FWD           7.23


Team total xMins summary (5 lowest, 5 highest):

                team_total  n_players
team                                 
Tunisia             967.40         26
Cabo Verde          967.70         26
Qatar               981.58         26
Congo DR            981.70         26
Korea Republic      982.06         26

...

             team_total  n_players
team                              
Norway           990.02         26
Morocco          990.02         26
Colombia         990.04         26
New Zealand      990.05         26
Uzbekistan       990.09         30


xMins distribution:

count    1256.00
mean       37.74
std        34.22
min         0.00
25%         6.83
50%        21.13
75%        76.52
max        90.00
Name: default_xmins, dtype: float64

## Manual Starters Validation

In [16]:
# === Preview function: validate XI + show full team xMins given those starters ===
from rapidfuzz import process, fuzz

def preview_manual_xi(team, proposed_players):
    """Validate proposed XI and preview the resulting xMins for all squad members."""
    fbref_squad = WC_TO_FBREF_SQUAD.get(team, team)
    team_data = squad_agg[squad_agg["Squad"] == fbref_squad].copy()
    team_roster = wc_roster[wc_roster["team"] == team].copy()

    if team_data.empty:
        print(f"⚠ No intl data for team '{team}'")
        return

    # Validation
    ascii_to_fbref = dict(zip(team_data["name_ascii"], team_data["Player"]))
    roster_pos    = dict(zip(team_roster["name_ascii"], team_roster["position"]))
    roster_player = dict(zip(team_roster["name_ascii"], team_roster["player"]))

    # Fallback for manual-XI starters with no intl data (uncapped backups, fresh
    # call-ups): mean conditional_min of actual starters at that position. Mirrors
    # build_default_xmins.py step 5 so the preview honors the locked XI instead of
    # dropping these players to the ~4-min leftover floor.
    _POS_NORM  = {"GK": "GK", "DF": "DEF", "MF": "MID", "FW": "FWD"}
    _POS_FBREF = {"GK": "GK", "DEF": "DF", "MID": "MF", "FWD": "FW"}
    _starters  = squad_agg[squad_agg["is_starter"]]
    _pos_norm  = _starters["Pos"].str.split(",").str[0].map(_POS_NORM)
    pos_avg_cond     = _starters["conditional_min"].groupby(_pos_norm).mean().to_dict()
    overall_avg_cond = _starters["conditional_min"].mean()

    proposed_ascii, unmatched, injected = [], [], []
    for p in proposed_players:
        a = to_ascii(MANUAL_OVERRIDES.get(p, p))
        if a in ascii_to_fbref:
            proposed_ascii.append(a)
        elif a in roster_pos:
            proposed_ascii.append(a)
            injected.append((p, a, roster_pos[a]))
        else:
            r = process.extractOne(a, list(ascii_to_fbref.keys()),
                                    scorer=fuzz.WRatio, score_cutoff=55)
            unmatched.append((p, ascii_to_fbref[r[0]] if r else None, r[1] if r else None))

    if injected:
        inj_rows = pd.DataFrame([{
            "name_ascii": a,
            "Player": roster_player[a],
            "Pos": _POS_FBREF.get(pos, "MF"),
            "is_gk": pos == "GK",
            "conditional_min": pos_avg_cond.get(pos, overall_avg_cond),
            "per_team_match": 0.0,
            "mp_share": 1.0,
        } for p, a, pos in injected])
        team_data = pd.concat([team_data, inj_rows], ignore_index=True)
        ascii_to_fbref.update(dict(zip(inj_rows["name_ascii"], inj_rows["Player"])))
        details = ", ".join(f"{p} ({pos}→{pos_avg_cond.get(pos, overall_avg_cond):.0f})"
                            for p, a, pos in injected)
        print(f"ℹ injected {len(injected)} no-intl-data starter(s) at "
              f"position-avg conditional_min: {details}")

    if unmatched:
        print(f"⚠ {len(unmatched)}/{len(proposed_players)} proposed players unmatched:")
        for name, sug, score in unmatched:
            sug_str = f"did you mean '{sug}'? (score {score:.0f})" if sug else "no close match"
            print(f"  ✗ {name:30s} → {sug_str}")

    if injected or unmatched:
        print("(preview below assumes the matched ones as starters)\n")

    # Override is_starter for this team
    team_data["is_starter"] = team_data["name_ascii"].isin(proposed_ascii)

    # Hybrid normalization, scoped to this team
    for is_gk_val in [True, False]:
        mask = team_data["is_gk"] == is_gk_val
        starters     = team_data[mask & team_data["is_starter"]]
        non_starters = team_data[mask & ~team_data["is_starter"]]
        budget = 90 if is_gk_val else 900
        sst = starters["conditional_min"].sum()
        starter_factor = min(1.0, budget / sst) if sst > 0 else 0
        ns_budget = budget - sst * starter_factor
        ns_sum = non_starters["per_team_match"].sum()
        ns_factor = min(1.0, ns_budget / ns_sum) if ns_sum > 0 else 0
        team_data.loc[mask & team_data["is_starter"], "xmins"] = (
            team_data.loc[mask & team_data["is_starter"], "conditional_min"] * starter_factor)
        team_data.loc[mask & ~team_data["is_starter"], "xmins"] = (
            team_data.loc[mask & ~team_data["is_starter"], "per_team_match"] * ns_factor)


    # Hybrid merge: name-only for unique names, position-aware for collisions (e.g., Danilos)
    POS_MAP = {"GK": "GK", "DF": "DEF", "MF": "MID", "FW": "FWD"}
    te = team_data.assign(pos_tokens=team_data["Pos"].str.split(",")).explode("pos_tokens")
    te["pos_primary"] = te["pos_tokens"].map(POS_MAP)
    te = te.drop_duplicates(subset=["Player", "pos_primary"])

    team_roster["dup_count"] = team_roster.groupby("name_ascii")["name_ascii"].transform("count")

    # Unique names → name-only merge (handles position discrepancy)
    te_namekey = te.drop_duplicates(subset="name_ascii")
    out_uniq = team_roster[team_roster["dup_count"] == 1].merge(
        te_namekey[["name_ascii", "is_starter", "xmins", "mp_share", "conditional_min"]],
        on="name_ascii", how="left"
    )

    # Duplicate names (Danilos) → position-aware merge to disambiguate
    out_dup = team_roster[team_roster["dup_count"] > 1].merge(
        te[["name_ascii", "pos_primary", "is_starter", "xmins", "mp_share", "conditional_min"]],
        left_on=["name_ascii", "position"], right_on=["name_ascii", "pos_primary"], how="left"
    ).drop_duplicates(subset=["player", "position"], keep="first")

    out = pd.concat([out_uniq, out_dup], ignore_index=True)
    out["xmins"] = out["xmins"].fillna(0)
    out["is_starter"] = out["is_starter"].fillna(False)
    out["mp_share"] = out["mp_share"].fillna(0)
    out["conditional_min"] = out["conditional_min"].fillna(0)

    leftover = max(0, 990 - out["xmins"].sum())
    boost = leftover / len(out) if len(out) > 0 else 0
    out["xmins"] = (out["xmins"] + boost).clip(upper=90)
    
    
    print(f"=== {team} preview ({len(out)} squad players, total xMins {out['xmins'].sum():.1f}) ===\n")
    sorted_out = out.sort_values(["is_starter", "xmins"], ascending=[False, False])
    cols = ["player", "position", "is_starter", "mp_share", "conditional_min", "xmins"]
    print(sorted_out[cols].round(2).to_string(index=False))

### Group A

Mexico

In [17]:
preview_manual_xi("Mexico", [
    "Raúl Rangel",   # GK
    "Israel Reyes",       # RB
    "Cesar Montes",       # CB
    "Johan Vasquez",        # CB
    "Jesus Gallardo",        # LB
    "Edson Alvarez",       # CDM
    "Erik Lira",   # CM
    "Álvaro Fidalgo",  # CM
    "Roberto Alvarado",       # RW
    "Raul Jimenez",        # ST
    "Alexis Vega",    # LW
])

ℹ injected 2 no-intl-data starter(s) at position-avg conditional_min: Raúl Rangel (GK→85), Álvaro Fidalgo (MID→73)

(preview below assumes the matched ones as starters)


=== Mexico preview (26 squad players, total xMins 990.0) ===


            player position is_starter  mp_share  conditional_min  xmins
       Raúl Rangel       GK       True      1.00            85.47  85.79
     Johan Vásquez      DEF       True      0.88            81.45  81.77
      César Montes      DEF       True      0.88            79.69  80.01
     Edson Álvarez      MID       True      0.82            79.12  79.45
  Roberto Alvarado      FWD       True      0.82            74.98  75.30
      Raúl Jiménez      FWD       True      0.74            74.70  75.02
    Jesús Gallardo      DEF       True      0.61            72.94  73.26
    Álvaro Fidalgo      MID       True      1.00            72.54  72.86
         Érik Lira      MID       True      0.37            70.66  70.98
       Alexis Vega      FWD       True      0.91            69.51  69.83
      Israel Reyes      DEF       True      0.46            65.20  65.52
     Jorge Sánchez      DEF      False      0.63            80.87  32.29
  Santiago Giménez      FWD      False      1.00   

Czech Republic

In [18]:
preview_manual_xi("Czechia", [
    "Matej Kovar",   # GK
    "Štěpán Chaloupek",       # RB
    "Robin Hranáč",       # CB
    "Ladislav Krejci",        # CB
    "Vladimir Coufal",        # LB
    "Tomas Soucek",       # CDM
    "Vladimír Darida",   # CM
    "David Jurásek",  # CM
    "Lukáš Provod",       # RW
    "Pavel Šulc",        # ST
    "Patrik Schick",    # LW
])

=== Czechia preview (26 squad players, total xMins 990.0) ===


          player position is_starter  mp_share  conditional_min  xmins
 Ladislav Krejcí      DEF       True      0.80            89.15  89.15
     Matej Kovár       GK       True      0.87            87.75  87.75
 Vladimír Coufal      DEF       True      0.94            86.70  86.70
    Tomás Soucek      MID       True      0.96            85.85  85.85
    Robin Hranác      DEF       True      0.39            84.38  84.38
      Pavel Sulc      MID       True      0.87            79.54  79.54
   Patrik Schick      FWD       True      0.66            75.48  75.48
   David Jurásek      DEF       True      0.25            74.56  74.56
Stepán Chaloupek      DEF       True      0.19            74.12  74.12
    Lukás Provod      MID       True      0.94            71.79  71.79
 Vladimír Darida      MID       True      0.13            68.43  68.43
      Lukás Cerv      MID      False      0.73            71.57  22.06
 Jaroslav Zeleny      DEF      False      0.60            80.94  21.64
     T

### Group B

Canada

In [19]:
preview_manual_xi("Canada", [
    "Maxime Crépeau",   # GK
    "Alistair Johnston",       # RB
    "Moïse Bombito",       # CB
    "Derek Cornelius",        # CB
    "Richie Laryea",        # LB
    "Tajon Buchanan",       # CDM
    "Ismael Kone",   # CM
    "Stephen Eustaquio",  # CM
    "Liam Millar",       # RW
    "Jonathan David",        # ST
    "Cyle Larin",    # LW
])

=== Canada preview (26 squad players, total xMins 990.0) ===


            player position is_starter  mp_share  conditional_min  xmins
    Maxime Crépeau       GK       True      0.55            80.85  80.85
     Moïse Bombito      DEF       True      0.52            80.62  80.62
 Stephen Eustaquio      MID       True      0.43            78.73  78.73
    Jonathan David      FWD       True      1.00            74.62  74.62
 Alistair Johnston      DEF       True      0.88            74.00  74.00
   Derek Cornelius      DEF       True      0.76            73.59  73.59
     Richie Laryea      DEF       True      0.88            72.02  72.02
       Ismaël Koné      MID       True      0.67            68.16  68.16
    Tajon Buchanan      FWD       True      0.74            64.18  64.18
       Liam Millar      FWD       True      0.52            61.50  61.50
        Cyle Larin      FWD       True      0.91            61.00  61.00
 Jacob Shaffelburg      MID      False      1.00            65.84  30.48
 Mathieu Choinière      MID      False      0.66   

Bosnia and Herzegovina

In [20]:
preview_manual_xi("Bosnia and Herzegovina", [
    "Nikola Vasilj",   # GK
    "Amar Dedic",       # RB
    "Nikola Katic",       # CB
    "Tarik Muharemovic",        # CB
    "Sead Kolasinac",        # LB
    "Ivan Bašić",       # CDM
    "Benjamin Tahirovic",   # CM
    "Kerim Alajbegović'",        # CM
    "Amar Memić",       # RW
    "Ermedin Demirović",        # ST
    "Edin Dzeko",    # LW
])

=== Bosnia and Herzegovina preview (26 squad players, total xMins 990.0) ===


             player position is_starter  mp_share  conditional_min  xmins
      Nikola Vasilj       GK       True      0.95            89.19  89.19
  Tarik Muharemovic      DEF       True      0.59            85.52  85.52
  Ermedin Demirovic      FWD       True      0.74            85.28  85.28
       Nikola Katic      DEF       True      0.69            83.72  83.72
         Amar Dedic      DEF       True      0.71            82.32  82.32
         Edin Dzeko      FWD       True      0.88            80.25  80.25
     Sead Kolasinac      DEF       True      0.38            75.61  75.61
 Benjamin Tahirovic      MID       True      1.00            71.61  71.61
         Amar Memic      MID       True      0.63            68.14  68.14
  Kerim Alajbegovic      FWD       True      0.49            61.17  61.17
         Ivan Basic      MID       True      0.74            49.55  49.55
        Ivan Sunjic      MID      False      0.66            79.45  31.92
      Dzenis Burnic      MID      Fals

Switzerland

In [21]:
preview_manual_xi("Switzerland", [
    "Gregor Kobel",   # GK
    "Silvan Widmer",       # RB
    "Manuel Akanji",       # CB
    "Nico Elvedi",        # CB
    "Ricardo Rodriguez",        # LB
    "Granit Xhaka",       # CDM
    "Remo Freuler",   # CM
    "Michel Aebischer",        # CM
    "Dan Ndoye",       # RW
    "Breel Embolo",        # ST
    "Ruben Vargas",    # LW
])

=== Switzerland preview (26 squad players, total xMins 990.0) ===


             player position is_starter  mp_share  conditional_min  xmins
      Manuel Akanji      DEF       True      0.89            86.32  86.33
       Remo Freuler      MID       True      0.84            84.97  84.98
       Gregor Kobel       GK       True      0.75            84.83  84.84
       Granit Xhaka      MID       True      0.94            84.15  84.16
          Dan Ndoye      FWD       True      0.70            82.38  82.39
        Nico Elvedi      DEF       True      0.64            82.19  82.20
  Ricardo Rodríguez      DEF       True      0.94            77.92  77.93
       Breel Embolo      FWD       True      0.94            75.40  75.41
      Silvan Widmer      DEF       True      0.74            72.97  72.98
   Michel Aebischer      MID       True      0.73            71.55  71.55
       Rubén Vargas      MID       True      0.78            70.75  70.76
      Fabian Rieder      MID      False      1.00            56.84  37.84
       Zeki Amdouni      FWD      Fals

### Group C

Brazil

In [22]:
preview_manual_xi("Brazil", [
    "Alisson",   # GK
    "Wesley",       # RB
    "Marquinhos",       # CB
    "Gabriel Magalhães",        # CB
    "Alex Sandro",        # LB
    "Bruno Guimaraes",       # CDM
    "Casemiro",   # CM
    "Matheus Cunha",        # CM
    "Raphinha",       # RW
    "Vinicius Júnior",        # ST
    "Luiz Henrique",    # LW
])

=== Brazil preview (26 squad players, total xMins 990.0) ===


            player position is_starter  mp_share  conditional_min  xmins
    Alisson Becker       GK       True      0.55            84.50  86.64
        Marquinhos      DEF       True      1.00            82.80  84.94
 Gabriel Magalhães      DEF       True      0.72            82.15  84.29
          Casemiro      MID       True      0.35            80.67  82.81
   Vinícius Júnior      MID       True      0.62            79.60  81.74
          Raphinha      MID       True      0.75            77.78  79.92
   Bruno Guimarães      MID       True      0.95            77.25  79.39
       Alex Sandro      DEF       True      0.10            76.86  79.00
            Wesley      DEF       True      0.15            70.88  73.02
     Matheus Cunha      FWD       True      0.35            51.50  53.64
     Luiz Henrique      FWD       True      0.40            51.31  53.45
            Danilo      DEF      False      0.50            75.47  39.99
     Lucas Paquetá      MID      False      0.45   

Morocco

In [67]:
preview_manual_xi("Morocco", [
    "Yassine Bounou",   # GK
    "Achraf Hakimi",       # RB
    "Nayef Aguerd",       # CB
    "Issa Diop",        # CB
    "Noussair Mazraoui",        # LB
    "Sofyan Amrabat",       # CDM
    "Neil El Aynaoui",   # CM
    "Azzedine Ounahi",        # CM
    "Brahim Diaz",       # RW
    "Ismael Saibari",        # ST
    "Abde Ezzalzouli",    # LW
])

ℹ injected 1 no-intl-data starter(s) at position-avg conditional_min: Issa Diop (DEF→79)


(preview below assumes the matched ones as starters)

=== Morocco preview (26 squad players, total xMins 990.0) ===

                     player position is_starter  mp_share  conditional_min  xmins
             Yassine Bounou       GK       True      1.00            88.16  88.23
               Nayef Aguerd      DEF       True      0.92            88.05  88.12
            Neil El Aynaoui      MID       True      0.61            87.44  87.51
              Achraf Hakimi      DEF       True      0.74            84.68  84.75
          Noussair Mazraoui      DEF       True      0.61            83.91  83.99
                Brahim Díaz      MID       True      0.84            80.68  80.75
             Sofyan Amrabat      MID       True      0.63            79.22  79.29
                  Issa Diop      DEF       True      1.00            78.87  78.94
             Ismael Saibari      MID       True      0.84            68.75  68.83
            Abde Ezzalzouli      FWD       True      0.69     

Scotland

In [69]:
preview_manual_xi("Scotland", [
    "Angus Gunn",   # GK
    "Aaron Hickey",       # RB
    "Scott McKenna",       # CB
    "John Souttar",        # CB
    "Andrew Robertson",        # LB
    "Lewis Ferguson",       # CDM
    "Scott McTominay",   # CM
    "Ben Gannon-Doak",        # CM
    "John McGinn",       # RW
    "Ryan Christie",        # ST
    "Che Adams",    # LW
])

=== Scotland preview (26 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
    Andy Robertson      DEF       True      1.00            85.39  85.39
   Scott McTominay      MID       True      1.00            84.82  84.82
        Angus Gunn       GK       True      0.59            83.70  83.70
    Lewis Ferguson      MID       True      0.43            82.50  82.50
       John McGinn      MID       True      0.88            80.73  80.73
      John Souttar      DEF       True      0.67            78.59  78.59
         Ché Adams      FWD       True      0.76            71.06  71.06
      Aaron Hickey      DEF       True      0.43            66.40  66.40
     Ryan Christie      MID       True      0.91            65.75  65.75
     Scott McKenna      DEF       True      0.74            65.18  65.18
   Ben Gannon-Doak      MID       True      0.87            61.31  61.31
      Grant Hanley      DEF      False      0.83            

### Group D

United States

In [25]:
preview_manual_xi("USA", [
    "Matt Freese",   # GK
    "Alexander Freeman",       # RB
    "Chris Richards",       # CB
    "Tim Ream",        # CB
    "Antonee Robinson",        # LB
    "Tyler Adams",       # CDM
    "Weston McKennie",   # CM
    "Malik Tillman",        # CM
    "Timothy Weah",       # RW
    "Folarin Balogun",        # ST
    "Christian Pulisic",    # LW
])

=== USA preview (26 squad players, total xMins 990.0) ===


             player position is_starter  mp_share  conditional_min  xmins
     Chris Richards      DEF       True      1.00            82.73  82.73
           Tim Ream      DEF       True      1.00            82.66  82.66
        Matt Freese       GK       True      0.74            81.85  81.85
  Alexander Freeman      DEF       True      0.74            81.62  81.62
   Antonee Robinson      DEF       True      0.26            78.46  78.46
  Christian Pulisic      MID       True      0.26            78.46  78.46
      Malik Tillman      MID       True      0.82            77.73  77.73
    Weston McKennie      MID       True      0.26            77.46  77.46
        Tyler Adams      MID       True      0.88            71.91  71.91
    Folarin Balogun      FWD       True      0.26            71.15  71.15
       Timothy Weah      MID       True      0.18            71.00  71.00
Sebastian Berhalter      MID      False      0.61            81.18  38.81
        Max Arfsten      FWD      Fals

Australia

In [73]:
preview_manual_xi("Australia", [
    "Mathew Ryan",       # GK
    "Jordy Bos",          # RB
    "Harry Souttar",       # CB
    "Alessandro Circati", # CB
    "Lucas Herrington",      # LB
    "Jacob Italiano",    # CDM
    "Jackson Irvine",       # CM
    "Aiden O'Neill",  # CM
    "Ajdin Hrustic",          # RW
    "Mo Toure",        # ST
    "Nestory Irankunda",    # LW
])

ℹ injected 2 no-intl-data starter(s) at position-avg conditional_min: Lucas Herrington (DEF→79), Jacob Italiano (DEF→79)
(preview below assumes the matched ones as starters)

=== Australia preview (26 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
       Mathew Ryan       GK       True      0.79            85.31  88.06
     Harry Souttar      DEF       True      0.79            85.31  88.06
Alessandro Circati      DEF       True      0.29            81.67  84.42
    Jackson Irvine      MID       True      1.00            81.42  84.17
  Lucas Herrington      DEF       True      1.00            78.87  81.62
    Jacob Italiano      DEF       True      1.00            78.87  81.62
     Aiden O'Neill      MID       True      0.71            65.53  68.28
 Nestory Irankunda      FWD       True      0.36            59.50  62.25
        Jordan Bos      DEF       True      0.64            57.79  60.54
     Ajdin Hrustic      MID   

Turkey

In [74]:
preview_manual_xi("Türkiye", [
    "Uğurcan Çakır",       # GK
    "Zeki Çelik",          # RB
    "Merih Demiral",       # CB
    "Abdülkerim Bardakcı", # CB
    "Ferdi Kadioglu",      # LB
    "Hakan Çalhanoglu",    # CDM
    "Orkun Kokcu",       # CM
    "Barış Alper Yılmaz",  # CM
    "Arda Güler",          # RW
    "Kenan Yıldız",        # ST
    "Kerem Aktürkoğlu",    # LW
])

=== Türkiye preview (35 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
Abdülkerim Bardakci      DEF       True      0.85            85.71  85.71
      Ugurcan Çakir       GK       True      0.57            84.40  84.40
         Arda Güler      MID       True      0.93            79.13  79.13
      Merih Demiral      DEF       True      0.78            77.21  77.21
   Hakan Çalhanoglu      MID       True      0.85            75.22  75.22
       Kenan Yildiz      MID       True      0.88            74.07  74.07
   Kerem Aktürkoglu      FWD       True      0.93            69.51  69.51
     Ferdi Kadioglu      DEF       True      0.81            69.30  69.30
        Orkun Kökçü      MID       True      0.90            65.15  65.15
         Zeki Çelik      DEF       True      0.68            63.48  63.48
 Baris Alper Yilmaz      FWD       True      0.70            61.95  61.95
        Mert Müldür      DEF      False      0.85

### Group E

Germany

In [78]:
preview_manual_xi("Germany", [
    "Manuel Neuer",   # GK
    "Joshua Kimmich",       # RB
    "Jonathan Tah",       # CB
    "Nico Schlotterbeck",        # CB
    "David Raum",        # LB
    "Aleksandar Pavlovic",       # CDM
    "Felix Nmecha",   # CM
    "Florian Wirtz",        # CM
    "Leroy Sane",       # RW
    "Jamal Musiala",        # ST
    "Kai Havertz",    # LW
])

=== Germany preview (27 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
     Joshua Kimmich      DEF       True      0.94            83.28  83.28
       Manuel Neuer       GK       True      0.16            82.00  82.00
       Jonathan Tah      DEF       True      0.88            78.17  78.17
      Florian Wirtz      MID       True      0.91            75.75  75.75
      Jamal Musiala      MID       True      0.43            75.74  75.74
        Kai Havertz      FWD       True      0.34            75.61  75.61
 Nico Schlotterbeck      DEF       True      0.53            73.09  73.09
         David Raum      DEF       True      0.66            72.67  72.67
Aleksandar Pavlovic      MID       True      0.44            70.30  70.30
         Leroy Sané      MID       True      0.52            66.73  66.73
       Felix Nmecha      MID       True      0.26            61.22  61.22
    Antonio Rüdiger      DEF      False      0.52

Cote d'Ivoire

In [80]:
preview_manual_xi("Côte d'Ivoire", [
    "Yahia Fofana",   # GK
    "Wilfried Singo",       # RB
    "Ousmane Diomande",       # CB
    "Obite N'Dicka",        # CB
    "Ghislain Konan",        # LB
    "Franck Kessie",       # CDM
    "Ibrahim Sangare",   # CM
    "Seko Fofana",        # CM
    "Amad Diallo",       # RW
    "Yan Diomande",        # ST
    "Ange-Yoan Bonny",    # LW
])

ℹ injected 1 no-intl-data starter(s) at position-avg conditional_min: Ange-Yoan Bonny (FWD→71)
(preview below assumes the matched ones as starters)

=== Côte d'Ivoire preview (26 squad players, total xMins 990.0) ===

           player position is_starter  mp_share  conditional_min  xmins
     Yahia Fofana       GK       True      0.87            85.54  85.54
      Evan Ndicka      DEF       True      0.73            84.93  84.93
   Ghislain Konan      DEF       True      0.58            83.75  83.75
    Franck Kessie      MID       True      1.00            77.85  77.85
      Amad Diallo      FWD       True      0.48            73.54  73.54
     Yan Diomande      FWD       True      0.41            72.35  72.35
 Ousmane Diomande      DEF       True      0.18            71.45  71.45
  Ibrahim Sangaré      MID       True      0.73            71.42  71.42
  Ange-Yoan Bonny      FWD       True      1.00            70.73  70.73
   Wilfried Singo      DEF       True      0.44            69.

Ecuador

In [81]:
preview_manual_xi("Ecuador", [
    "Hernan Galindez",   # GK
    "Joel Ordóñez",       # RB
    "Willian Pacho",       # CB
    "Piero Hincapie",        # CB
    "Alan Franco",        # LB
    "Moises Caicedo",       # CDM
    "Pedro Vite",   # CM
    "Pervis Estupinan",        # CM
    "Gonzalo Plata",       # RW
    "John Yeboah",        # ST
    "Enner Valencia",    # LW
])

=== Ecuador preview (26 squad players, total xMins 990.0) ===

          player position is_starter  mp_share  conditional_min  xmins
   Willian Pacho      DEF       True      1.00            87.00  87.00
 Hernán Galíndez       GK       True      0.60            85.59  85.59
  Moisés Caicedo      MID       True      0.90            84.96  84.96
  Piero Hincapié      DEF       True      0.80            83.71  83.71
Pervis Estupiñán      DEF       True      0.65            82.94  82.94
    Joel Ordóñez      DEF       True      0.40            82.00  82.00
     Alan Franco      MID       True      0.75            80.20  80.20
      Pedro Vite      MID       True      0.45            76.43  76.43
   Gonzalo Plata      FWD       True      0.45            72.00  72.00
  Enner Valencia      FWD       True      0.82            71.02  71.02
     John Yeboah      MID       True      0.52            51.19  51.19
    Félix Torres      DEF      False      0.85            73.05  34.85
 Ángelo Precia

### Group F

Netherlands

In [82]:
preview_manual_xi("Netherlands", [
    "Bart Verbruggen",   # GK
    "Denzel Dumfries",       # RB
    "Jurrien Timber",       # CB
    "Virgil Van Dijk",        # CB
    "Micky Van de Ven",        # LB
    "Ryan Gravenberch",       # CDM
    "Tijjani Reijnders",   # CM
    "Frenkie de Jong",        # CM
    "Donyell Malen",       # RW
    "Memphis",        # ST
    "Cody Gakpo",    # LW
])

=== Netherlands preview (26 squad players, total xMins 990.0) ===

               player position is_starter  mp_share  conditional_min  xmins
      Bart Verbruggen       GK       True      0.84            87.14  87.14
      Virgil van Dijk      DEF       True      0.92            86.91  86.91
      Denzel Dumfries      DEF       True      0.72            83.86  83.86
      Frenkie de Jong      MID       True      0.55            79.06  79.06
           Cody Gakpo      FWD       True      1.00            77.87  77.87
    Tijjani Reijnders      MID       True      0.90            75.28  75.28
     Ryan Gravenberch      MID       True      0.61            74.73  74.73
        Memphis Depay      FWD       True      0.75            72.68  72.68
       Jurriën Timber      DEF       True      0.43            66.50  66.50
     Micky van de Ven      DEF       True      0.63            66.04  66.04
        Donyell Malen      FWD       True      0.80            56.93  56.93
           Nathan Aké

Japan

In [84]:
preview_manual_xi("Japan", [
    "Zion Suzuki",   # GK
    "Takehiro Tomiyasu",       # RB
    "Hiroki Ito",       # CB
    "Ko Itakura",        # CB
    "Ritsu Doan",        # LB
    "Kaishu Sano",       # CDM
    "Wataru Endo",   # CM
    "Keito Nakamura",        # CM
    "Takefusa Kubo",       # RW
    "Daichi Kamada",        # ST
    "Ayase Ueda",    # LW
])

=== Japan preview (26 squad players, total xMins 990.0) ===

           player position is_starter  mp_share  conditional_min  xmins
      Zion Suzuki       GK       True      0.83            85.00  85.00
       Ko Itakura      DEF       True      0.92            79.12  79.12
       Hiroki Ito      DEF       True      0.50            79.09  79.09
Takehiro Tomiyasu      DEF       True      0.17            77.14  77.14
      Wataru Endo      MID       True      0.92            77.12  77.12
       Ayase Ueda      FWD       True      0.75            75.71  75.71
      Kaishu Sano      MID       True      0.25            71.62  71.62
       Ritsu Doan      DEF       True      1.00            64.47  64.47
    Takefusa Kubo      MID       True      0.92            64.25  64.25
    Daichi Kamada      MID       True      1.00            60.76  60.76
   Keito Nakamura      MID       True      0.83            52.80  52.80
  Shogo Taniguchi      DEF      False      0.67            75.15  37.49
   

Sweden

In [ ]:
preview_manual_xi("Sweden", [
    "Kristoffer Nordfeldt",   # GK
    "Isak Hien",       # RB
    "Carl Starfelt",       # CB
    "Victor Lindelof",        # CB
    "Gustaf Lagerbielke",        # LB
    "Yasin Ayari",       # CDM
    "Jesper Karlstrom",   # CM
    "Gabriel Gudmundsson",        # CM
    "Anthony Elanga",       # RW
    "Viktor Gyokeres",        # ST
    "Alexander Isak",    # LW
])

=== Sweden preview (26 squad players, total xMins 990.0) ===

                  player position is_starter  mp_share  conditional_min  xmins
         Viktor Gyökeres      FWD       True      0.84            84.42  84.42
               Isak Hien      DEF       True      0.78            81.10  81.10
    Kristoffer Nordfeldt       GK       True      0.16            79.29  79.29
     Gabriel Gudmundsson      DEF       True      0.94            77.13  77.13
      Gustaf Lagerbielke      DEF       True      0.49            75.64  75.64
             Yasin Ayari      MID       True      0.92            74.15  74.15
         Victor Lindelöf      DEF       True      0.44            73.63  73.63
        Jesper Karlström      MID       True      0.50            72.68  72.68
           Carl Starfelt      DEF       True      0.42            72.36  72.36
          Alexander Isak      FWD       True      0.56            71.75  71.75
          Anthony Elanga      FWD       True      0.61            66.

### Group G

Belgium

In [86]:
preview_manual_xi("Belgium", [
    "Thibaut Courtois",   # GK
    "Thomas Meunier",       # RB
    "Zeno Debast",       # CB
    "Arthur Theate",        # CB
    "Maxim De Cuyper",        # LB
    "Amadou Onana",       # CDM
    "Youri Tielemans",   # CM
    "Kevin de Bruyne",        # CM
    "Charles de Ketelaere",       # RW
    "Leandro Trossard",        # ST
    "Jeremy Doku",    # LW
])

=== Belgium preview (26 squad players, total xMins 990.0) ===

                player position is_starter  mp_share  conditional_min  xmins
      Thibaut Courtois       GK       True      0.28            81.67  81.74
       Kevin De Bruyne      MID       True      0.66            79.44  79.52
           Jérémy Doku      MID       True      0.90            79.14  79.22
           Zeno Debast      DEF       True      0.79            78.36  78.43
         Arthur Theate      DEF       True      0.92            76.52  76.59
          Amadou Onana      MID       True      0.64            72.70  72.77
       Youri Tielemans      MID       True      0.73            68.26  68.34
      Leandro Trossard      MID       True      0.80            65.56  65.64
       Maxim De Cuyper      DEF       True      0.69            65.37  65.45
  Charles De Ketelaere      MID       True      0.58            63.36  63.44
        Thomas Meunier      DEF       True      0.54            57.61  57.68
      Timothy

### Group H

Spain

In [87]:
preview_manual_xi("Spain", [
    "Unai Simon",   # GK
    "Marcos Llorente",       # RB
    "Pau Cubarsi",       # CB
    "Aymeric Laporte",        # CB
    "Marc Cucurella",        # LB
    "Rodri",       # CDM
    "Pedri",   # CM
    "Fabián Ruiz",        # CM
    "Lamine Yamal",       # RW
    "Mikel Oyarzabal",        # ST
    "Nico Williams",    # LW
])

=== Spain preview (26 squad players, total xMins 990.0) ===

            player position is_starter  mp_share  conditional_min  xmins
        Unai Simón       GK       True      0.72            88.93  88.93
    Marc Cucurella      DEF       True      0.78            86.93  86.93
   Aymeric Laporte      DEF       True      0.58            84.31  84.31
      Lamine Yamal      MID       True      0.63            77.83  77.83
       Fabián Ruiz      MID       True      0.73            75.67  75.67
     Nico Williams      MID       True      0.64            73.03  73.03
       Pau Cubarsí      DEF       True      0.37            70.93  70.93
             Rodri      MID       True      0.35            67.53  67.53
   Marcos Llorente      DEF       True      0.18            65.75  65.75
   Mikel Oyarzabal      FWD       True      0.92            64.94  64.94
             Pedri      MID       True      0.79            60.27  60.27
  Martín Zubimendi      MID      False      0.81            73.

Uruguay

In [89]:
preview_manual_xi("Uruguay", [
    "Sergio Rochet",   # GK
    "Guillermo Varela",       # RB
    "Ronald Araujo",       # CB
    "Sebastian Caceres",        # CB
    "Mathias Olivera",        # LB
    "Manuel Ugarte",       # CDM
    "Federico Valverde",   # CM
    "Rodrigo Bentancur",        # CM
    "Agustin Canobbio",       # RW
    "Maximiliano Araújo",        # ST
    "Darwin Nunez",    # LW
])

=== Uruguay preview (26 squad players, total xMins 990.0) ===

                player position is_starter  mp_share  conditional_min  xmins
         Sergio Rochet       GK       True      0.85            86.59  86.59
     Federico Valverde      MID       True      0.90            83.72  83.72
         Ronald Araujo      DEF       True      0.50            81.57  81.57
         Manuel Ugarte      MID       True      0.95            78.60  78.60
           Maxi Araújo      MID       True      0.90            76.87  76.87
          Darwin Núñez      FWD       True      0.80            75.24  75.24
       Mathías Olivera      DEF       True      0.72            74.21  74.21
     Sebastián Cáceres      DEF       True      0.72            72.82  72.82
     Rodrigo Bentancur      MID       True      0.60            70.12  70.12
      Guillermo Varela      DEF       True      0.45            64.00  64.00
      Agustín Canobbio      FWD       True      0.18            53.12  53.12
     Facundo 

### Group I

France

In [36]:
preview_manual_xi("France", [
    "Mike Maignan",  # GK
    "Jules Kounde",       # RB
    "William Saliba",         # CB
    "Dayot Upamecano",         # CB
    "Theo Hernandez",      # LB
    "Aurélien Tchouaméni",     # CDM
    "Adrien Rabiot",      # CM
    "Desire Doue",      # CM
    "Michael Olise", # RW
    "Kylian Mbappe",     # ST
    "Ousmane Dembele",    # LW]               
])

=== France preview (26 squad players, total xMins 990.0) ===


              player position is_starter  mp_share  conditional_min  xmins
        Mike Maignan       GK       True      0.94            88.05  88.12
      William Saliba      DEF       True      0.68            87.55  87.62
      Theo Hernández      DEF       True      0.59            87.26  87.34
     Dayot Upamecano      DEF       True      0.68            85.54  85.61
       Kylian Mbappé      FWD       True      0.67            83.55  83.63
 Aurélien Tchouaméni      MID       True      0.50            82.80  82.88
        Jules Koundé      DEF       True      0.79            82.60  82.68
     Ousmane Dembélé      MID       True      0.52            72.14  72.22
       Michael Olise      MID       True      0.66            69.28  69.35
       Adrien Rabiot      MID       True      0.56            65.19  65.26
         Désiré Doué      MID       True      0.19            63.86  63.94
           Manu Koné      MID      False      0.60            75.10  19.13
     Ibrahima Konaté     

Senegal

In [90]:
preview_manual_xi("Senegal", [
    "Edouard Mendy",  # GK
    "Krepin Diatta",       # RB
    "Kalidou Koulibaly",         # CB
    "Moussa Niakhate",         # CB
    "El Hadji Malick Diouf",      # LB
    "Idrissa Gana Gueye",     # CDM
    "Pape Gueye",      # CM
    "Iliman Ndiaye",      # CM
    "Habib Diarra", # RW
    "Sadio Mane",     # ST
    "Nicolas Jackson",    # LW]               
])

=== Senegal preview (29 squad players, total xMins 989.9) ===

               player position is_starter  mp_share  conditional_min  xmins
        Édouard Mendy       GK       True      0.93            87.14  90.00
      Moussa Niakhaté      DEF       True      0.82            85.00  87.91
        Krépin Diatta      MID       True      0.82            84.01  86.92
    Kalidou Koulibaly      DEF       True      0.91            82.62  85.52
El Hadji Malick Diouf      DEF       True      0.50            81.93  84.84
           Sadio Mané      MID       True      0.87            81.78  84.68
        Iliman Ndiaye      FWD       True      0.82            74.74  77.65
           Pape Gueye      MID       True      0.62            73.72  76.63
      Nicolas Jackson      FWD       True      0.70            63.83  66.74
         Habib Diarra      MID       True      0.46            63.65  66.56
      Pape Matar Sarr      MID      False      0.72            68.79  26.37
         Ismaïla Sarr    

Norway

In [91]:
preview_manual_xi("Norway", [
    "Ørjan Nyland",  # GK
    "Julian Ryerson",       # RB
    "Kristoffer Ajer",         # CB
    "Torbjørn Heggem",         # CB
    "David Møller Wolfe",      # LB
    "Sander Berge",     # CDM
    "Fredrik Aursnes",      # CM
    "Martin Ødegaard",      # CM
    "Alexander Sørloth", # RW
    "Erling Haaland",     # ST
    "Antonio Nusa",    # LW]               
])

ℹ injected 1 no-intl-data starter(s) at position-avg conditional_min: Fredrik Aursnes (MID→73)
(preview below assumes the matched ones as starters)

=== Norway preview (26 squad players, total xMins 990.0) ===

              player position is_starter  mp_share  conditional_min  xmins
        Ørjan Nyland       GK       True      0.89            85.25  85.25
      Erling Haaland      FWD       True      1.00            84.34  84.34
      Julian Ryerson      DEF       True      1.00            84.13  84.13
     Kristoffer Ajer      DEF       True      0.77            83.54  83.54
        Sander Berge      MID       True      1.00            82.83  82.83
  David Møller Wolfe      DEF       True      0.89            82.13  82.13
     Martin Ødegaard      MID       True      0.52            81.86  81.86
        Antonio Nusa      MID       True      0.84            75.34  75.34
   Alexander Sørloth      FWD       True      1.00            74.74  74.74
     Torbjørn Heggem      DEF       Tru

### Group J

Austria

In [94]:
preview_manual_xi("Austria", [
    "Alexander Schlager",   # GK
    "Konrad Laimer",       # RB
    "Philipp Lienhart",       # CB
    "David Alaba",        # CB
    "Stefan Posch",        # LB
    "Nicolas Seiwald",       # CDM
    "Xaver Schlager",   # CM
    "Patrick Wimmer",        # CM
    "Michael Gregoritsch",       # RW
    "Marcel Sabitzer",        # ST
    "Marko Arnautovic",    # LW
])

=== Austria preview (26 squad players, total xMins 990.0) ===

               player position is_starter  mp_share  conditional_min  xmins
      Nicolas Seiwald      MID       True      1.00            85.36  85.36
      Marcel Sabitzer      MID       True      0.95            84.86  84.86
   Alexander Schlager       GK       True      0.45            83.42  83.42
     Philipp Lienhart      DEF       True      0.85            81.29  81.29
        Konrad Laimer      MID       True      1.00            79.58  79.58
          David Alaba      DEF       True      0.28            74.67  74.67
         Stefan Posch      DEF       True      0.93            73.08  73.08
     Marko Arnautovic      FWD       True      0.88            64.94  64.94
       Xaver Schlager      MID       True      0.35            64.10  64.10
  Michael Gregoritsch      FWD       True      0.85            55.03  55.03
       Patrick Wimmer      MID       True      0.75            52.77  52.77
Christoph Baumgartner    

Argentina

In [92]:
preview_manual_xi("Argentina", [
    "Emiliano Martinez",   # GK
    "Nahuel Molina",       # RB
    "Cristian Romero",       # CB
    "Nicolas Otamendi",        # CB
    "Nicolas Tagliafico",        # LB
    "Enzo Fernandez",   # CM
    "Alexis Mac Allister",        # CM
    "Rodrigo de Paul",       # RW
    "Thiago Almada",
    "Lionel Messi",        # ST
    "Julian Alvarez",    # LW
])

=== Argentina preview (26 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
  Emiliano Martínez       GK       True      0.90            87.50  87.51
    Cristian Romero      DEF       True      0.83            83.33  83.34
     Enzo Fernández      MID       True      0.83            80.78  80.79
    Rodrigo De Paul      MID       True      0.93            80.33  80.33
 Nicolás Tagliafico      DEF       True      0.86            80.20  80.20
       Lionel Messi      FWD       True      0.69            78.54  78.55
   Nicolás Otamendi      DEF       True      0.93            77.59  77.60
      Nahuel Molina      DEF       True      0.88            76.55  76.56
Alexis Mac Allister      MID       True      0.79            76.16  76.17
     Julián Alvarez      FWD       True      0.98            73.45  73.46
      Thiago Almada      MID       True      0.33            70.92  70.92
   Lautaro Martínez      FWD      False      0.

### Group K

Portugal

In [95]:
preview_manual_xi("Portugal", [
    "Diogo Costa",   # GK
    "Joao Cancelo",       # RB
    "Ruben Dias",       # CB
    "Goncalo Inacio",        # CB
    "Nuno Mendes",        # LB
    "Vitinha",       # CDM
    "Joao Neves",   # CM
    "Bruno Fernandes",        # CM
    "Bernardo Silva",       # RW
    "Pedro Neto",        # ST
    "Cristiano Ronaldo",    # LW
])

=== Portugal preview (27 squad players, total xMins 990.0) ===

             player position is_starter  mp_share  conditional_min  xmins
        Diogo Costa       GK       True      0.95            89.85  89.85
         Rúben Dias      DEF       True      0.88            88.87  88.87
        Nuno Mendes      DEF       True      0.84            87.19  87.19
    Bruno Fernandes      MID       True      0.86            85.32  85.32
            Vitinha      MID       True      0.88            80.24  80.24
  Cristiano Ronaldo      FWD       True      0.89            79.51  79.51
     Bernardo Silva      MID       True      0.86            77.60  77.60
       João Cancelo      DEF       True      0.48            77.13  77.13
     Gonçalo Inácio      DEF       True      0.48            70.06  70.06
         João Neves      MID       True      0.64            68.56  68.56
         Pedro Neto      MID       True      0.67            68.18  68.18
        Rúben Neves      MID      False      0.7

Colombia

In [96]:
preview_manual_xi("Colombia", [
    "Camilo Vargas",   # GK
    "Daniel Munoz",       # RB
    "Davinson Sanchez",       # CB
    "Jhon Lucumi",        # CB
    "Johan Mojica",        # LB
    "Jefferson Lerma",       # CDM
    "Richard Rios",   # CM
    "Jhon Arias",        # CM
    "James Rodriguez",       # RW
    "Luis Diaz",        # ST
    "Luis Suarez",    # LW
])

ℹ injected 1 no-intl-data starter(s) at position-avg conditional_min: Luis Suarez (FWD→71)
(preview below assumes the matched ones as starters)

=== Colombia preview (26 squad players, total xMins 990.0) ===

                player position is_starter  mp_share  conditional_min  xmins
         Camilo Vargas       GK       True      0.81            87.09  87.23
      Dávinson Sánchez      DEF       True      0.71            85.10  85.24
          Daniel Muñoz      DEF       True      0.74            84.78  84.92
           Jhon Lucumí      DEF       True      0.69            84.49  84.62
          Johan Mojica      DEF       True      0.52            84.47  84.61
             Luis Díaz      MID       True      0.95            83.74  83.88
       Jefferson Lerma      MID       True      0.79            77.77  77.90
       James Rodríguez      MID       True      1.00            70.98  71.12
           Luis Suárez      FWD       True      1.00            70.73  70.86
            Jhon Aria

### Group L

England

In [97]:
preview_manual_xi("England", [
    "Jordan Pickford",   # GK
    "Reece James",       # RB
    "John Stones",       # CB
    "Marc Guehi",        # CB
    "Nico O'Reilly",        # LB
    "Declan Rice",       # CDM
    "Elliot Anderson",   # CM
    "Jude Bellingham",        # CM
    "Bukayo Saka",       # RW
    "Harry Kane",        # ST
    "Anthony Gordon",    # LW
])

=== England preview (26 squad players, total xMins 990.0) ===

          player position is_starter  mp_share  conditional_min  xmins
 Jordan Pickford       GK       True      0.93            87.63  87.63
      Marc Guéhi      DEF       True      0.63            81.01  81.01
     John Stones      DEF       True      0.62            80.80  80.80
 Jude Bellingham      MID       True      0.75            80.26  80.26
      Harry Kane      FWD       True      1.00            79.80  79.80
   Nico O'Reilly      DEF       True      0.13            79.29  79.29
     Declan Rice      MID       True      0.95            76.16  76.16
 Elliot Anderson      MID       True      0.33            74.50  74.50
     Bukayo Saka      MID       True      0.57            74.45  74.45
     Reece James      DEF       True      0.40            71.00  71.00
  Anthony Gordon      MID       True      0.60            57.24  57.24
      Ezri Konsa      DEF      False      0.66            70.47  27.95
   Morgan Roge

Croatia

In [99]:
preview_manual_xi("Croatia", [
    "Dominik Livakovic",   # GK
    "Josip Stanisic",       # RB
    "Josip Sutalo",       # CB
    "Luka Vuskovic",        # CB
    "Josko Gvardiol",        # LB
    "Luka Modric",       # CDM
    "Mateo Kovacic",   # CM
    "Petar Sucic",        # CM
    "Andrej Kramaric",       # RW
    "Ante Budimir",        # ST
    "Ivan Perisic",    # LW
])

=== Croatia preview (26 squad players, total xMins 990.0) ===

           player position is_starter  mp_share  conditional_min  xmins
Dominik Livakovic       GK       True      0.89            86.49  86.49
     Josip Sutalo      DEF       True      0.80            84.21  84.21
   Josko Gvardiol      DEF       True      0.80            80.11  80.11
   Josip Stanisic      DEF       True      0.49            79.92  79.92
    Mateo Kovacic      MID       True      0.44            75.75  75.75
  Andrej Kramaric      FWD       True      1.00            70.12  70.12
      Petar Sucic      MID       True      0.72            69.33  69.33
      Luka Modric      MID       True      1.00            68.12  68.12
    Luka Vuskovic      DEF       True      0.13            67.00  67.00
     Ivan Perisic      FWD       True      1.00            64.10  64.10
     Ante Budimir      FWD       True      0.71            54.05  54.05
  Duje Caleta-Car      DEF      False      0.72            81.54  35.99
 